## Random Forest (A bagging Technique)

In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [3]:
df=pd.read_csv('data/raw/Travel.csv')

In [72]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [71]:
df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(),inplace=True)

C:\Users\awais\AppData\Local\Temp\ipykernel_3000\4293906714.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(),inplace=True)


In [69]:
df['NumberOfChildrenVisiting']

0       0.0
1       2.0
2       0.0
3       1.0
4       0.0
       ... 
4883    1.0
4884    2.0
4885    3.0
4886    2.0
4887    2.0
Name: NumberOfChildrenVisiting, Length: 4888, dtype: float64

In [27]:
df['ProductPitched']=df['ProductPitched'].str.replace('King','Super Deluxe')

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4888 non-null   float64
 3   TypeofContact             4888 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4888 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4888 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4888 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4888 non-null   float64
 14  Passport

In [78]:
df.drop('CustomerID',axis=1,inplace=True)

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ProdTaken                 4888 non-null   int64  
 1   Age                       4888 non-null   float64
 2   TypeofContact             4888 non-null   object 
 3   CityTier                  4888 non-null   int64  
 4   DurationOfPitch           4888 non-null   float64
 5   Occupation                4888 non-null   object 
 6   Gender                    4888 non-null   object 
 7   NumberOfPersonVisiting    4888 non-null   int64  
 8   NumberOfFollowups         4888 non-null   float64
 9   ProductPitched            4888 non-null   object 
 10  PreferredPropertyStar     4888 non-null   float64
 11  MaritalStatus             4888 non-null   object 
 12  NumberOfTrips             4888 non-null   float64
 13  Passport                  4888 non-null   int64  
 14  PitchSat

In [81]:
df['total_visits']=df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']

In [85]:
df.drop(['NumberOfPersonVisiting','NumberOfChildrenVisiting'],axis=1,inplace=True)

In [86]:
num_features=[col for col in df.columns if df[col].dtype != 'o']
print(f'Numerical Feature: {len(num_features)}')

Numerical Feature: 18


### Train test Split

In [90]:
from sklearn.model_selection import train_test_split
X=df.drop('ProdTaken',axis=1)
y=df['ProdTaken']

In [91]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

In [92]:
X_train.shape,X_test.shape

((3666, 17), (1222, 17))

In [93]:
cat_features=X.select_dtypes(include='object').columns
num_features=X.select_dtypes(exclude='object').columns

In [97]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer=OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder',oh_transformer,cat_features),
        ('StandardScalar',numeric_transformer,num_features),
    ]
)

In [98]:
X_train=preprocessor.fit_transform(X_train)

In [101]:
X_train=pd.DataFrame(X_train)
X_train

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,...,-0.721708,-0.653318,0.277912,1.777611,2.053422,1.575272,0.681958,-1.273702,-0.415942,-0.058810
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,-0.721708,-0.165525,-0.723883,-0.724971,-0.670111,-0.634811,1.409353,0.785113,-0.224146,-0.768009
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.721708,-0.287473,1.279708,0.526320,1.508716,-0.634811,0.681958,0.785113,-0.711215,-0.768009
3,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.454995,-0.531370,0.277912,1.777611,-0.670111,-0.634811,1.409353,0.785113,0.057482,-0.058810
4,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,-0.721708,0.322268,-0.723883,-0.724971,-0.670111,1.575272,0.681958,0.785113,-1.139911,-0.058810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3661,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,...,-0.721708,-0.653318,1.279708,-0.724971,-0.670111,-0.634811,-1.500228,0.785113,-0.531928,0.650390
3662,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.454995,-0.897214,-0.723883,1.777611,-1.214818,-0.634811,1.409353,0.785113,1.528543,-0.058810
3663,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.454995,1.541750,0.277912,-0.724971,2.053422,-0.634811,-0.772833,0.785113,-0.356053,0.650390
3664,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.454995,1.785647,1.279708,-0.724971,-0.125404,-0.634811,-1.500228,0.785113,-0.248595,0.650390


In [99]:
X_test=preprocessor.transform(X_test)

### Model-Training 

In [106]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,roc_auc_score,recall_score,f1_score,precision_score

In [111]:
models={
    'Random_Forest':RandomForestClassifier(),
    'Decision_Tree':DecisionTreeClassifier()
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    model_train_accuracy= accuracy_score(y_train,y_train_pred)
    model_train_f1= f1_score(y_train,y_train_pred)
    model_train_precision= precision_score(y_train,y_train_pred)
    model_train_recall= recall_score(y_train,y_train_pred)
    model_train_rocauc= roc_auc_score(y_train,y_train_pred)
    
    model_test_accuracy= accuracy_score(y_test,y_test_pred)
    model_test_f1= f1_score(y_test,y_test_pred)
    model_test_precision= precision_score(y_test,y_test_pred)
    model_test_recall= recall_score(y_test,y_test_pred)
    model_test_rocauc= roc_auc_score(y_test,y_test_pred)
    
   
    
    print('--------------------------------')
    print(f'Accuracy Score: {model_train_accuracy}')
    print(f'F1 Score: {model_train_f1}')
    print(f'Precicion Score: {model_train_precision}')
    print(f'Recall Score: {model_train_recall}')
    print(f'ROCAUC Score: {model_train_rocauc}')
    
    print('--------------------------------')
    print(f'Accuracy Score: {model_test_accuracy}')
    print(f'F1 Score: {model_test_f1}')
    print(f'Precicion Score: {model_test_precision}')
    print(f'Recall Score: {model_test_recall}')
    print(f'ROCAUC Score: {model_test_rocauc}')

--------------------------------
Accuracy Score: 1.0
F1 Score: 1.0
Precicion Score: 1.0
Recall Score: 1.0
ROCAUC Score: 1.0
--------------------------------
Accuracy Score: 0.9328968903436988
F1 Score: 0.7819148936170213
Precicion Score: 0.9607843137254902
Recall Score: 0.6591928251121076
ROCAUC Score: 0.8265934095530508
--------------------------------
Accuracy Score: 1.0
F1 Score: 1.0
Precicion Score: 1.0
Recall Score: 1.0
ROCAUC Score: 1.0
--------------------------------
Accuracy Score: 0.9198036006546645
F1 Score: 0.7850877192982456
Precicion Score: 0.7682403433476395
Recall Score: 0.8026905829596412
ROCAUC Score: 0.8743182644527935
